# 02. Input data inventory

Before any of the source datasets are parsed, this notebook opens each one and
records what it actually contains: the files present, their size, the layers or
tables inside them, the field names and types, the number of records, and the
coordinate reference system where there is geometry.

Two purposes. The structure of every input is documented in one place, so that
the parsing code written against these datasets can be read alongside a record
of what it was written for. And the record is produced by opening the files
rather than by describing them from memory, so a file that has been replaced,
truncated or re-exported in a different form is visible immediately instead of
causing a confusing failure later.

Nothing is parsed, joined or transformed here. The notebook reads headers and
metadata only, and at most a handful of rows from any file.

The datasets covered are the 2020 population census at block and tract level,
the national geostatistical framework block geometries, the 2020 urban
environment street audit, the national economic unit directory, the two
deprivation indices at tract level, the metropolitan area boundaries, and the
OpenStreetMap extracts.

Output is `Results/02_input_inventory.parquet`, one row per file, together with
a full run report written to `Results/logs/` recording the field list of every
dataset.

Runtime is a few minutes. Nothing is cached because nothing is expensive.

## 1. Setup

In [ ]:
import sys, os, glob, zipfile, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'module is pointing at {mx.BASE}'

import numpy as np
import pandas as pd
import pyogrio

rep = mx.Report('02_input_inventory').start()

INPUTS = mx.DIRS['inputs']
print('inputs root:', INPUTS)

## 2. What is present

Each dataset is a directory under `Inputs/`. The scan records every file with
its size, and stops if a directory expected to hold data is missing or empty,
since an empty directory otherwise produces an empty result further down the
pipeline without any error being raised.

In [ ]:
DATASETS = {
    'census':       f'{INPUTS}/Census',
    'mgn_manzanas': f'{INPUTS}/MGN/manzanas',
    'cleu':         f'{INPUTS}/CLEU2020',
    'denue':        f'{INPUTS}/DENUE',
    'deprivation':  f'{INPUTS}/Deprivation',
    'metro_areas':  f'{INPUTS}/metro_areas',
    'osm':          f'{INPUTS}/osm',
}

rows = []
for name, d in DATASETS.items():
    if not os.path.isdir(d):
        raise FileNotFoundError(f'{name}: directory does not exist: {d}')
    found = sorted(p for p in glob.glob(os.path.join(d, '**', '*'), recursive=True)
                   if os.path.isfile(p))
    if not found:
        raise FileNotFoundError(f'{name}: directory is empty: {d}')
    for p in found:
        rows.append({
            'dataset': name,
            'file': os.path.relpath(p, INPUTS),
            'ext': os.path.splitext(p)[1].lower(),
            'mb': round(os.path.getsize(p) / 1e6, 2),
            'path': p,
        })

FILES = pd.DataFrame(rows)
print(FILES.groupby('dataset')
           .agg(files=('file', 'size'), mb=('mb', 'sum'),
                extensions=('ext', lambda s: ', '.join(sorted(set(s)))))
           .round(1).to_string())
print(f'\ntotal: {len(FILES)} files, {FILES.mb.sum():,.0f} MB')

## 3. Reading the structure of a file

Each kind of file is opened in the way appropriate to it. Archives are listed
and the header of their largest tabular member is read. Spatial files report
their layers, fields, record count and projection through their metadata rather
than by loading geometry. Delimited text and spreadsheets have a few rows read
so that field names and apparent types can be recorded.

Everything is wrapped so that a file which cannot be opened records the reason
and the scan continues, because knowing which file failed and why is more
useful than stopping at the first one.

In [ ]:
def _peek_text(path, nrows=3, **kw):
    """Field names and dtypes from the first rows of a delimited text file."""
    for enc in ('utf-8', 'latin1'):
        try:
            df = pd.read_csv(path, nrows=nrows, dtype=str, encoding=enc,
                             low_memory=False, **kw)
            return df, enc
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1, f'cannot decode {path}')


def probe(path):
    """Describe one file. Returns (summary dict, detail DataFrame or None)."""
    ext = os.path.splitext(path)[1].lower()
    base = os.path.basename(path)

    if ext == '.zip':
        with zipfile.ZipFile(path) as zf:
            members = [(i.filename, i.file_size) for i in zf.infolist()
                       if not i.is_dir()]
        tabular = sorted([m for m in members
                          if m[0].lower().endswith(('.csv', '.txt'))],
                         key=lambda m: -m[1])
        det = pd.DataFrame(members, columns=['member', 'bytes'])
        summary = {'kind': 'zip archive', 'n_members': len(members),
                   'detail': f'{len(tabular)} tabular members'}
        if tabular:
            with zipfile.ZipFile(path) as zf, zf.open(tabular[0][0]) as fh:
                df, enc = _peek_text(fh)
            summary['detail'] = (f'largest tabular member {tabular[0][0]}, '
                                 f'{df.shape[1]} fields, encoding {enc}')
            summary['fields'] = list(df.columns)
        return summary, det

    if ext in ('.gpkg', '.shp', '.geojson', '.gdb'):
        layers = [r[0] for r in pyogrio.list_layers(path)]
        det_rows = []
        fields = None
        for lyr in layers:
            info = pyogrio.read_info(path, layer=lyr)
            if fields is None:
                fields = list(info['fields'])
            det_rows.append({'layer': lyr,
                             'features': info.get('features'),
                             'geometry': info.get('geometry_type'),
                             'crs': info.get('crs'),
                             'n_fields': len(info['fields'])})
        det = pd.DataFrame(det_rows)
        return ({'kind': 'spatial', 'n_members': len(layers),
                 'detail': f"layers: {', '.join(layers)}",
                 'fields': fields}, det)

    if ext in ('.csv', '.txt', '.tsv'):
        sep = '\t' if ext == '.tsv' else ','
        df, enc = _peek_text(path, sep=sep)
        return ({'kind': 'delimited text', 'n_members': 1,
                 'detail': f'{df.shape[1]} fields, encoding {enc}',
                 'fields': list(df.columns)},
                pd.DataFrame({'field': df.columns,
                              'first_value': df.iloc[0].values if len(df) else None}))

    if ext in ('.xlsx', '.xls', '.xlsm'):
        xl = pd.ExcelFile(path)
        det = pd.DataFrame({'sheet': xl.sheet_names})
        head = xl.parse(xl.sheet_names[0], header=None, nrows=8, dtype=str)
        return ({'kind': 'spreadsheet', 'n_members': len(xl.sheet_names),
                 'detail': f"sheets: {', '.join(xl.sheet_names)}",
                 'fields': None}, head)

    if ext == '.pbf':
        return ({'kind': 'OSM extract', 'n_members': None,
                 'detail': 'binary, not opened here', 'fields': None}, None)

    return ({'kind': 'other', 'n_members': None, 'detail': '', 'fields': None}, None)


def describe(dataset, limit=1):
    """Probe up to `limit` representative files of one dataset and print them."""
    sub = FILES[FILES.dataset == dataset].sort_values('mb', ascending=False)
    print(f'=== {dataset}: {len(sub)} files, {sub.mb.sum():,.1f} MB ===\n')
    print(sub[['file', 'ext', 'mb']].head(12).to_string(index=False))
    if len(sub) > 12:
        print(f'   ... and {len(sub) - 12} more')

    out = []
    for p in sub['path'].head(limit):
        print(f'\n--- {os.path.basename(p)} ---')
        try:
            summary, det = probe(p)
        except Exception as exc:
            print(f'    could not open: {type(exc).__name__}: {exc}')
            out.append({'dataset': dataset, 'file': os.path.basename(p),
                        'kind': 'unreadable', 'detail': str(exc)[:200]})
            continue
        print(f"    kind    : {summary['kind']}")
        print(f"    detail  : {summary['detail']}")
        if summary.get('fields'):
            print(f"    fields  : {len(summary['fields'])}")
            for j in range(0, len(summary['fields']), 6):
                print('              ' + ', '.join(summary['fields'][j:j + 6]))
        if det is not None and len(det):
            print('    structure:')
            print('      ' + det.head(25).to_string(index=False).replace('\n', '\n      '))
            if len(det) > 25:
                print(f'      ... and {len(det) - 25} more rows')
        rec = {'dataset': dataset, 'file': os.path.basename(p)}
        rec.update({k: v for k, v in summary.items() if k != 'fields'})
        rec['n_fields'] = len(summary['fields']) if summary.get('fields') else None
        rec['fields'] = '; '.join(summary['fields']) if summary.get('fields') else None
        out.append(rec)
        if det is not None and len(det):
            rep.add_table(f'{dataset}: {os.path.basename(p)}', det, max_rows=60)
    print()
    return out

## 4. Population census

Block and tract level results of the 2020 census, distributed as one archive per
federal entity. The field list of the largest tabular member identifies the
population and dwelling counts the pipeline needs.

In [ ]:
probes = []
probes += describe('census', limit=2)

## 5. Block geometries

The national geostatistical framework, which supplies the block polygons used as
population-weighted origins for travel time.

In [ ]:
probes += describe('mgn_manzanas', limit=3)

## 6. Urban environment street audit

The 2020 block level audit recording, side by side along each block, the
presence of sidewalks, street vendors and other features of the walking
environment.

In [ ]:
probes += describe('cleu', limit=1)

## 7. Economic unit directory

The establishment register from which food outlets are selected. The field list
determines how outlets are classified by activity code, name and geography.

In [ ]:
probes += describe('denue', limit=2)

## 8. Deprivation indices

Two tract level measures of deprivation, which differ in coverage and in the
direction their scales run.

In [ ]:
probes += describe('deprivation', limit=2)

## 9. Metropolitan area boundaries

One boundary file per metropolitan area, used to clip every other layer.

In [ ]:
probes += describe('metro_areas', limit=2)

ma = FILES[FILES.dataset == 'metro_areas'].sort_values('file')
print('all metropolitan area boundary files:')
for j in range(0, len(ma), 3):
    print('   ' + '   '.join(os.path.basename(f) for f in ma['file'].iloc[j:j + 3]))

## 10. OpenStreetMap extracts

The source from which the pedestrian and road networks were built. Listed for
provenance only.

In [ ]:
probes += describe('osm', limit=1)

## 11. Checks

In [ ]:
print('--- checks ---\n')

INV = pd.DataFrame(probes)

# Fails if any representative file could not be opened at all.
bad = INV[INV['kind'] == 'unreadable']
print(f'1. files that could not be opened               {len(bad)}')
if len(bad):
    print(bad[['dataset', 'file', 'detail']].to_string(index=False))

# Fails if a dataset present on disk yielded no readable structure, which would
# mean the inventory looks complete while describing nothing.
described = set(INV.loc[INV['kind'] != 'unreadable', 'dataset'])
undescribed = sorted(set(DATASETS) - described - {'osm'})
print(f'2. datasets with no readable structure          {len(undescribed)}')
assert not undescribed, undescribed

# The census is distributed one archive per federal entity, so a count short of
# 32 means an entity is missing and part of the country would be absent from the
# tract table without anything being raised.
n_census = int((FILES.dataset == 'census').sum())
print(f'3. census archives                              {n_census} of 32')
assert n_census == 32, f'{n_census} census archives found'

# One boundary file per metropolitan area is expected, and the count is reported
# against the 92 areas in the crosswalk.
metros = mx.load_metros()
n_bounds = int((FILES.dataset == 'metro_areas').sum())
print(f'4. metropolitan area boundary files             {n_bounds} for {len(metros)} areas')

## 12. Write results

In [ ]:
out = os.path.join(mx.DIRS['results'], '02_input_inventory.parquet')
FILES.drop(columns='path').to_parquet(out, index=False)
print('written:', out)

out2 = os.path.join(mx.DIRS['results'], '02_input_structure.parquet')
INV.to_parquet(out2, index=False)
print('written:', out2)

rep.add_table('all input files', FILES.drop(columns='path'), max_rows=250)
rep.add_table('structure of representative files', INV, max_rows=40)
rep.note(f'{len(FILES)} input files, {FILES.mb.sum():,.0f} MB across '
         f'{FILES.dataset.nunique()} datasets')
rep.save()